#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Create Flag Parameter

In [0]:
# Check whether the Gold dimension table already exists
if spark.catalog.tableExists("store_catalog.gold.dim_erp_PX_CAT_G1V2"):
    incremental_flag = "1"
else:
    incremental_flag = "0"

In [0]:
print(incremental_flag)

1


#Create Dimension erp_PX_CAT_G1V2

###Fetch Relative Columns

In [0]:
# Load distinct product records from the Silver layer
df_scr = spark.sql('''
        SELECT DISTINCT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_PX_CAT_G1V2`
    ''')


In [0]:
df_scr.limit(10).display()

category_id,category,subcategory,maintenance_flag,record_priority
AC_BR,Accessories,Bike Racks,true,1
CO_DE,Components,Derailleurs,true,1
AC_PU,Accessories,Pumps,true,1
CL_JE,Clothing,Jerseys,false,1
AC_LO,Accessories,Locks,true,1
CO_CS,Components,Cranksets,true,1
AC_HE,Accessories,Helmets,true,1
CO_WH,Components,Wheels,true,1
CL_SO,Clothing,Socks,false,1
CO_RF,Components,Road Frames,true,1


###dim_erp_PX_CAT_G1V2- Initial and Incremental

In [0]:
# Read existing Gold records or create an empty DataFrame for the initial load
if spark.catalog.tableExists("store_catalog.gold.dim_erp_PX_CAT_G1V2") == True:
    df_sink = spark.sql('''
            SELECT
                erp_PX_CAT_G1V2_surrogate_key,
                category_id,
                category,
                subcategory,
                maintenance_flag
            FROM
                store_catalog.gold.dim_erp_PX_CAT_G1V2
        ''')

else:
    df_sink = spark.sql('''
            SELECT
                1 as erp_PX_CAT_G1V2_surrogate_key,
                category_id,
                category,
                subcategory,
                maintenance_flag
            FROM
                parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_PX_CAT_G1V2`
            WHERE
                1 = 0
        ''')


###Filtering new records and old records

In [0]:
# Match Silver records with existing Gold records by product ID
df_filter = df_scr.join(df_sink, df_scr["category_id"] == df_sink["category_id"], "left")\
        .select(df_scr["category_id"],df_scr["category"],df_scr["subcategory"],df_scr["maintenance_flag"]
        ,df_sink["erp_PX_CAT_G1V2_surrogate_key"]
    )

**df_filter_old**

In [0]:
# Separate existing product records
df_filter_old = df_filter.filter(col("erp_PX_CAT_G1V2_surrogate_key").isNotNull())

**df_filter_new**

In [0]:
# Select new products that require surrogate keys
df_filter_new = df_filter.filter(col("erp_PX_CAT_G1V2_surrogate_key").isNull())\
    .select(df_scr["category_id"],df_scr["category"],df_scr["subcategory"],df_scr["maintenance_flag"]
    )

In [0]:
df_filter_new.limit(10).display()

category_id,category,subcategory,maintenance_flag


###Create Surrogate Key

**Fetch the max Surrogate Key from existing table**

In [0]:
# Retrieve the next available surrogate key
if (incremental_flag == "0"):
    max_value = 1
else:
    max_value_df = spark.sql('''SELECT MAX(erp_PX_CAT_G1V2_surrogate_key) FROM store_catalog.gold.dim_erp_PX_CAT_G1V2''')
    max_value = max_value_df.collect()[0][0]+1

**Create Surrogate Key column and Add the max surrogate key**

In [0]:
# Assign unique surrogate keys to new product records
df_filter_new = df_filter_new.withColumn("erp_PX_CAT_G1V2_surrogate_key",max_value + monotonically_increasing_id())
df_filter_new.limit(10).display()

category_id,category,subcategory,maintenance_flag,erp_PX_CAT_G1V2_surrogate_key


###Create Final DF - df_filter_old + df_filter_new

In [0]:
# Combine new customers with existing product records
df_final = df_filter_new.union(df_filter_old)

In [0]:
df_final.limit(10).display()

category_id,category,subcategory,maintenance_flag,erp_PX_CAT_G1V2_surrogate_key
AC_BR,Accessories,Bike Racks,true,1
CO_DE,Components,Derailleurs,true,2
AC_PU,Accessories,Pumps,true,3
CL_JE,Clothing,Jerseys,false,4
AC_LO,Accessories,Locks,true,5
CO_CS,Components,Cranksets,true,6
AC_HE,Accessories,Helmets,true,7
CO_WH,Components,Wheels,true,8
CL_SO,Clothing,Socks,false,9
CO_RF,Components,Road Frames,true,10


# SCD TYPE - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("store_catalog.gold.dim_erp_PX_CAT_G1V2"):
    # Incremental Run
    # Merge product records into the Gold dimension table
    delta_tbl = DeltaTable.forPath(spark,"abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_erp_PX_CAT_G1V2")

    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.erp_PX_CAT_G1V2_surrogate_key = src.erp_PX_CAT_G1V2_surrogate_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()

else:
    # Initial Run
    # Create the Gold dimension table during the initial load
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_erp_PX_CAT_G1V2")\
        .saveAsTable("store_catalog.gold.dim_erp_PX_CAT_G1V2")


In [0]:
%sql
SELECT * FROM store_catalog.gold.dim_erp_PX_CAT_G1V2 LIMIT 10

category_id,category,subcategory,maintenance_flag,erp_PX_CAT_G1V2_surrogate_key
AC_BR,Accessories,Bike Racks,true,1
CO_DE,Components,Derailleurs,true,2
AC_PU,Accessories,Pumps,true,3
CL_JE,Clothing,Jerseys,false,4
AC_LO,Accessories,Locks,true,5
CO_CS,Components,Cranksets,true,6
AC_HE,Accessories,Helmets,true,7
CO_WH,Components,Wheels,true,8
CL_SO,Clothing,Socks,false,9
CO_RF,Components,Road Frames,true,10


In [0]:
%sql
SELECT COUNT(*) FROM store_catalog.gold.dim_erp_PX_CAT_G1V2 

count(1)
37
